In [ ]:
"""
tvn_notconnect4.py
==============
Task Vector Negation (TVN) para Connect 4 → Not Connect4.

Corrección principal respecto a versión anterior:
  El mapeo de claves ahora es dinámico: detecta automáticamente el formato
  real de las claves del adaptador en lugar de asumir un patrón fijo.
  Esto resuelve el error "Vectores de tarea calculados: 0 capas" que ocurría
  porque load_peft_weights devuelve claves sin ".default." (formato raw del
  safetensors), mientras que PeftModel.state_dict() sí las incluye.

Dependencias:
  pip install transformers peft torch huggingface_hub
"""

import os
import gc
import torch
import shutil
import logging
import argparse
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import load_peft_weights

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def detectar_formato_claves(lora_state_dict: dict, base_state_dict: dict) -> str:
    """
    Detecta automáticamente el formato de las claves del adaptador
    probando transformaciones hasta encontrar una que haga match.

    Retorna el primer base_key que hace match, o lanza error si ninguno funciona.
    """
    # Tomar la primera clave lora_B disponible como muestra
    muestra = next((k for k in lora_state_dict if "lora_B" in k), None)
    if muestra is None:
        raise RuntimeError("No se encontró ninguna clave lora_B en el adaptador.")

    logger.info(f"Clave lora_B de muestra: {muestra}")

    # Lista de transformaciones a probar en orden
    transformaciones = [
        # Formato B: base_model.model. sin .default.  ← más común con load_peft_weights
        lambda k: k.replace("base_model.model.", "").replace(".lora_B.weight", ".weight"),
        # Formato A: base_model.model. con .default.
        lambda k: k.replace("base_model.model.", "").replace(".lora_B.default.weight", ".weight"),
        # Formato C: sin base_model, sin .default.
        lambda k: k.replace(".lora_B.weight", ".weight"),
        # Formato D: sin base_model, con .default.
        lambda k: k.replace(".lora_B.default.weight", ".weight"),
        # Formato E: base_model.model.model. (doble model) sin .default.
        lambda k: k.replace("base_model.model.model.", "model.").replace(".lora_B.weight", ".weight"),
        # Formato F: base_model.model.model. (doble model) con .default.
        lambda k: k.replace("base_model.model.model.", "model.").replace(".lora_B.default.weight", ".weight"),
    ]

    for i, transform in enumerate(transformaciones):
        base_key = transform(muestra)
        if base_key in base_state_dict:
            logger.info(f"Formato detectado (transformación {i+1}): {muestra} → {base_key}")
            return transform  # Retorna la función de transformación que funcionó

    # Ninguna funcionó — imprimir claves para diagnóstico manual
    logger.error("No se pudo detectar el formato de claves automáticamente.")
    logger.error("=== Claves del adaptador (primeras 8) ===")
    for k in list(lora_state_dict.keys())[:8]:
        logger.error(f"  LORA: {k}")
    logger.error("=== Claves del modelo base (primeras 8) ===")
    for k in list(base_state_dict.keys())[:8]:
        logger.error(f"  BASE: {k}")
    raise RuntimeError(
        "No se encontró transformación válida. "
        "Revisa las claves impresas arriba y ajusta la función detectar_formato_claves."
    )


def main(args):
    logger.info("=== Iniciando TVN (Connect 4 / Not Connect 4) ===")
    logger.info(f"Modelo Base     : {args.base_model}")
    logger.info(f"Adaptador LoRA  : {args.lora_model}")
    logger.info(f"Alpha a aplicar : {args.alpha}")
    logger.info(f"Escala LoRA     : {args.lora_alpha_param}/{args.lora_r_param} = {args.lora_alpha_param/args.lora_r_param}")

    token = os.getenv("HF_TOKEN")

    # Intentar obtener el token en orden de prioridad:
    # 1. Variable de entorno HF_TOKEN
    # 2. Archivo de credenciales de HuggingFace (~/.cache/huggingface/token)
    # 3. HF_HOME personalizado (/workspace/.cache/huggingface/token)
    token = os.getenv("HF_TOKEN")

    if not token:
        # Buscar en las ubicaciones estándar del token guardado por `hf auth login`
        candidatos = [
            os.path.expanduser("~/.cache/huggingface/token"),
            os.path.join(os.getenv("HF_HOME", ""), "token"),
            "/workspace/.cache/huggingface/token",
        ]
        for ruta in candidatos:
            if ruta and os.path.exists(ruta):
                with open(ruta) as f:
                    token = f.read().strip()
                if token:
                    logger.info(f"Token leído desde: {ruta}")
                    break

    if not token:
        logger.warning("No se encontró HF_TOKEN. La subida al Hub fallará.")
    else:
        logger.info("Token de HuggingFace disponible.")

    # =========================================================================
    # 1. Cargar modelo base y tokenizador
    # =========================================================================
    logger.info("Cargando modelo base en CPU...")
    base_model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype       = torch.bfloat16,
        device_map        = "cpu",
        low_cpu_mem_usage = True,
        token             = token,
    )
    tokenizer = AutoTokenizer.from_pretrained(args.base_model, token=token)
    base_state_dict = base_model.state_dict()

    # =========================================================================
    # 2. Cargar pesos LoRA
    # =========================================================================
    logger.info("Cargando pesos del adaptador...")
    lora_state_dict = load_peft_weights(args.lora_model, token=token)
    # Mover todos los pesos LoRA a CPU explícitamente.
    # load_peft_weights puede colocarlos en GPU si hay una disponible,
    # causando OOM al multiplicar B @ A junto al modelo base en memoria.
    lora_state_dict = {k: v.cpu() for k, v in lora_state_dict.items()}
    logger.info("Pesos LoRA movidos a CPU.")

    scaling = args.lora_alpha_param / args.lora_r_param
    logger.info(f"Escala efectiva (lora_alpha / r): {scaling:.4f}")

    # =========================================================================
    # 3. Detectar formato de claves automáticamente
    # =========================================================================
    key_transform = detectar_formato_claves(lora_state_dict, base_state_dict)

    # =========================================================================
    # 4. Calcular vectores de tarea
    # =========================================================================
    pristine_weights = {}
    task_vectors     = {}
    skipped          = 0

    for key in list(lora_state_dict.keys()):
        if "lora_B" not in key:
            continue

        base_key   = key_transform(key)
        lora_A_key = key.replace("lora_B", "lora_A")

        if base_key not in base_state_dict:
            logger.debug(f"base_key no encontrado: {base_key}")
            skipped += 1
            continue

        if lora_A_key not in lora_state_dict:
            logger.debug(f"lora_A no encontrada: {lora_A_key}")
            skipped += 1
            continue

        W_base = base_state_dict[base_key].clone().float().cpu()
        W_A    = lora_state_dict[lora_A_key].float().cpu()   # (r, d_in)
        W_B    = lora_state_dict[key].float().cpu()           # (d_out, r)

        # τ = (B @ A) * scaling  →  (d_out, d_in)
        # Operación en CPU: el modelo base ya está en CPU (device_map="cpu")
        # y los pesos LoRA pueden cargarse en GPU por defecto; .cpu() lo previene.
        tau = (W_B @ W_A) * scaling

        pristine_weights[base_key] = W_base
        task_vectors[base_key]     = tau

    logger.info(f"Vectores de tarea calculados: {len(task_vectors)} capas")
    logger.info(f"Claves omitidas             : {skipped}")

    if len(task_vectors) == 0:
        raise RuntimeError("task_vectors vacío tras la detección automática. Revisa los logs.")

    del lora_state_dict
    gc.collect()

    # =========================================================================
    # 5. Aplicar Alpha único
    # =========================================================================
    alpha = args.alpha
    logger.info(f"\n--- Procesando Alpha: {alpha} ---")

    # Aplicar TVN: W_tvn = W_base - alpha × τ
    for base_key in task_vectors:
        W_tvn = pristine_weights[base_key] - (alpha * task_vectors[base_key])
        base_state_dict[base_key].copy_(W_tvn.to(torch.bfloat16))

    # =========================================================================
    # 6. Guardar localmente
    # =========================================================================
    temp_dir = f"./temp_tvn_notconnect4_alpha_{str(alpha).replace('.', '_')}"
    os.makedirs(temp_dir, exist_ok=True)
    base_model.save_pretrained(temp_dir, max_shard_size="5GB")
    tokenizer.save_pretrained(temp_dir)
    logger.info(f"Guardado localmente en: {temp_dir}")

    # =========================================================================
    # 7. Subir a Hugging Face Hub
    # =========================================================================
    repo_id = f"{args.repo_prefix}-alpha-{str(alpha).replace('.', '_')}"
    logger.info(f"Subiendo a Hub: {repo_id}...")
    try:
        base_model.push_to_hub(repo_id, token=token, private=False)
        tokenizer.push_to_hub(repo_id, token=token)
        logger.info(f"✓ Subido con éxito: {repo_id}")
    except Exception as e:
        logger.error(f"✗ Error al subir {repo_id}: {e}")

    # =========================================================================
    # 8. Limpiar carpeta temporal
    # =========================================================================
    shutil.rmtree(temp_dir)
    logger.info(f"Carpeta temporal eliminada.")

    logger.info("\n=== ¡Proceso de TVN Completado! ===")
    logger.info(f"Repo generado en Hub: {repo_id}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="TVN para Connect 4 → NotConnect4."
    )
    parser.add_argument("--base_model",       type=str,   default="Qwen/Qwen3-14B")
    parser.add_argument("--lora_model",       type=str,   required=True)
    parser.add_argument("--repo_prefix",      type=str,   required=True)
    parser.add_argument("--alpha",            type=float, required=True)
    parser.add_argument("--lora_alpha_param", type=int,   default=128)
    parser.add_argument("--lora_r_param",     type=int,   default=64)
    args = parser.parse_args()
    main(args)

    # Uso:
    #   python tvn_notconnect4.py \
    #     --lora_model  HailNicol/qwen3-14b-connect4-ftl-v3 \
    #     --repo_prefix HailNicol/qwen3-14b-notconnect4 \
    #     --alpha 0.5